In [86]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt

In [87]:
ticker_file = pd.read_csv('Tickers_Example.csv', header=None)
ticker_file.rename(columns={0: 'NAME'}, inplace=True)

In [88]:
def clean_data(tickers):
    filtered_stocks = pd.DataFrame()
    start_date = "2023-10-01"
    end_date = "2024-09-30"
    for ticker in tickers['NAME']:
        try:
            stock = yf.Ticker(ticker)
            info = stock.fast_info
            if info['currency'] not in ["USD", "CAD"]: # Enusring stock is listed, traded in CAD or USD
                continue
    
            hist = stock.history(start=start_date, end=end_date, interval="1d")
            hist['Month'] = hist.index.to_period('M')
            monthly_data = hist.groupby('Month').filter(lambda x: len(x) >= 18)

            avg_monthly_volume = monthly_data.groupby('Month')['Volume'].mean().mean()
            if avg_monthly_volume >= 100000:
                filtered_stocks = pd.concat([filtered_stocks, pd.DataFrame({"Ticker": [ticker]})])
                
        except Exception as e:
            continue
    
    return filtered_stocks.reset_index(drop=True)

In [89]:
ticker_file = clean_data(ticker_file)

/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_45000/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_45000/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_45000/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/7x/z5l3v4x13gv5wlh0qdjwgllh0000gn/T/ipykernel_45000/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
$AGN: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
/var/folde

In [90]:
def getBeta(cur_ticker):
    SP_ticker = '^GSPC'

    cur_stock = yf.Ticker(cur_ticker)
    SP_index = yf.Ticker(SP_ticker)

    # Will edit these later
    start_date = '2022-11-14'
    end_date = '2024-11-14'

    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)
    SP_index_hist = SP_index.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = pd.DataFrame(cur_stock_hist['Close'])
    prices['S&P Index'] = SP_index_hist['Close']

    daily_returns = prices.pct_change(fill_method=None).dropna()
    daily_returns.drop(index=daily_returns.index[0], inplace=True)

    SP_var = daily_returns['S&P Index'].var()
    SP_beta = daily_returns.cov() / SP_var

    return SP_beta.iat[0,1]

In [91]:
# Gets average of daily growth
def getGrowth(cur_ticker):
    # Fetch data for current stock
    cur_stock = yf.Ticker(cur_ticker)
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = cur_stock_hist['Close']
    
    daily_returns = prices.pct_change(fill_method=None).dropna()
    SP_growth = daily_returns.mean()
    
    return SP_growth[cur_ticker] * 100

In [92]:
def getVolatility(cur_ticker):
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    stock_data = yf.Ticker(cur_ticker).history(start=start_date, end=end_date)
    stock_data['Daily Return'] = stock_data['Close'].pct_change(fill_method=None).dropna()
    
    volatility = stock_data['Daily Return'].std()
    return volatility

In [93]:
stock_data = []

for i in range(len(ticker_file)):
    cur_ticker = ticker_file['Ticker'].iloc[i]

    beta = getBeta(cur_ticker)
    growth = getGrowth(cur_ticker)
    volatility = getVolatility(cur_ticker)

    stock_details = {
        "name": cur_ticker,
        "beta": beta,
        "growth": growth,
        "volatility": volatility
    }

    stock_data = stock_data + [stock_details]

for i in range(len(stock_data)):
    print(stock_data[i])

{'name': 'AAPL', 'beta': 1.1194903168018533, 'growth': 0.09539631726542762, 'volatility': 0.014246559461932467}
{'name': 'ABBV', 'beta': 0.2115726299426246, 'growth': 0.04736297625604738, 'volatility': 0.013500496584209496}
{'name': 'ABT', 'beta': 0.4541755554735703, 'growth': 0.040384450225892175, 'volatility': 0.012154342039766541}
{'name': 'ACN', 'beta': 0.9728216138211596, 'growth': 0.0669812706466094, 'volatility': 0.014994612780564841}
{'name': 'AIG', 'beta': 0.8383566584211121, 'growth': 0.06643425268893459, 'volatility': 0.014746088289175684}
{'name': 'AMZN', 'beta': 1.5277748093342405, 'growth': 0.1736770062572734, 'volatility': 0.01947138887810965}
{'name': 'AXP', 'beta': 1.0770037766497065, 'growth': 0.14202532872325013, 'volatility': 0.01579195249787853}
{'name': 'BA', 'beta': 0.8949164029335099, 'growth': -0.0234808083421831, 'volatility': 0.019634197749691013}
{'name': 'BAC', 'beta': 1.0090966132085066, 'growth': 0.06251821807790532, 'volatility': 0.015906188298297593}
{'